# 00 · Setup and environment

This is the first notebook in a series that follows Fast-dLLM v2 (1.5B) from its eager Hugging Face
form to the compiled TVM Relax artifact the runners load, and checks what changes along the way.

| # | Notebook | Model | Question it answers |
|---|---|---|---|
| 00 | Setup and environment | none | Can this machine run the rest? |
| 01 | The eager HF model | real 1.5B | What does the reference compute, step by step? |
| 02 | From HF to `relax.frontend.nn` | tiny + real config | How is the model rewritten, and what does TVM need from it? |
| 03 | Numerical parity | real weights | Where and how much do the numbers differ? |
| 04 | The pipeline, pass by pass | tiny | What does each compiler pass do to the IR? |
| 05 | Memory planning and VM lowering | tiny (+ real artifact) | How does the IR become VM bytecode? |
| 06 | Runtime and decode loop | real artifact | How does the compiled model actually generate text? |
| 07 | Profiling | real artifact | Where does the time go? |
| 08 | Other targets | tiny | How do the Metal and Jetson builds differ? |

Every notebook starts with the same cell. `bootstrap()` puts the in-tree TVM build on `sys.path`,
sets `TVM_LIBRARY_PATH` and nvcc's directory, and changes the working directory to the repo root. It
must run before anything imports `tvm`.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

## Build and tool versions

In [2]:
print_table([{"item": k, "value": v} for k, v in env_report().items()])

item                value                                    
------------------  -----------------------------------------
repo commit         00eccbd                                  
tvm version         0.24.dev0                                
tvm commit          a7bfc857b                                
USE_CUDA            ON                                       
USE_CUBLAS          ON                                       
USE_CUTLASS         ON                                       
USE_METAL           OFF                                      
metal codegen       True                                     
cuda device         NVIDIA RTX 4500 Ada Generation           
compute capability  8.9                                      
nvcc                Build cuda_12.8.r12.8/compiler.35404655_0
torch               2.10.0+cu128                             
transformers        4.53.1                                   
aarch64 cross g++   True                                     


The TVM submodule is pinned to a commit and carries one local patch. Here is what the patch touches
(it has to be applied before building TVM; see `scripts/setup_tvm.sh`):

In [3]:
import subprocess
print(subprocess.run(["git", "apply", "--stat", "patches/tvm-a7bfc857b.patch"], capture_output=True, text=True).stdout)

 cmake/modules/CUDA.cmake                        |   24 ++++++++++
 python/tvm/contrib/cutlass/gemm_operation.py    |    2 -
 python/tvm/contrib/cutlass/gen_tensor_op.py     |    2 -
 python/tvm/relax/backend/cuda/cublas.py         |    7 +++
 python/tvm/s_tir/dlight/gpu/matmul.py           |   20 +++++---
 python/tvm/topi/nn/batch_matmul.py              |    7 ++-
 src/arith/iter_affine_map.cc                    |   49 ++++++++++++++++++++
 src/relax/backend/contrib/codegen_c/codegen_c.h |   36 +++++++++++++++
 src/target/opt/build_metal_off.cc               |   57 ++++++++++++++++++++++-
 9 files changed, 191 insertions(+), 13 deletions(-)



## Checkpoint and artifacts

In [4]:
from pathlib import Path
from huggingface_hub import try_to_load_from_cache
from fast_dllm_relax.common import DEFAULT_MODEL_NAME, DEFAULT_MODEL_REVISION

cached = try_to_load_from_cache(DEFAULT_MODEL_NAME, "model.safetensors", revision=DEFAULT_MODEL_REVISION)
print(f"{DEFAULT_MODEL_NAME}@{DEFAULT_MODEL_REVISION[:8]} cached locally: {isinstance(cached, str)}")

rows = []
for d in sorted(Path("artifacts").iterdir()):
    names = d / "fast_dllm_relax_nn_param_names.txt"
    if not names.exists():
        continue
    lines = names.read_text().splitlines()
    dtype_file = d / "fast_dllm_relax_nn_dtype.txt"
    rows.append({
        "artifact": rel(d),
        "dtype": dtype_file.read_text().strip() if dtype_file.exists() else "?",
        "params": len(lines),
        "has lm_head": any("lm_head" in n for n in lines),
        "npz MB": round((d / "fast_dllm_relax_nn_params.npz").stat().st_size / 1e6),
    })
print_table(rows)

Efficient-Large-Model/Fast_dLLM_v2_1.5B@25093b6f cached locally: True
artifact                                    dtype     params  has lm_head  npz MB
------------------------------------------  --------  ------  -----------  ------
artifacts/fast_dllm_relax_bf16              bfloat16  198     False        3087  
artifacts/fast_dllm_relax_bf16_jetson_orin  bfloat16  198     False        3087  
artifacts/fast_dllm_relax_fp16              float16   198     False        3087  
artifacts/fast_dllm_relax_fp16_jetson_orin  float16   198     False        3087  
artifacts/fast_dllm_relax_fp16_metal        float16   198     False        3087  


## Which notebooks can run here

In [5]:
env = env_report()
cuda = bool(env["cuda device"])
ckpt = isinstance(cached, str)
art = Path("artifacts/fast_dllm_relax_bf16/fast_dllm_relax_nn.so").exists()
print_table([
    {"notebook": "01 eager HF", "needs": "CUDA + checkpoint", "ready": cuda and ckpt},
    {"notebook": "02 frontend", "needs": "TVM (+ checkpoint for the real param table)", "ready": True},
    {"notebook": "03 parity", "needs": "CUDA + checkpoint + bf16 artifact", "ready": cuda and ckpt and art},
    {"notebook": "04 / 05 pipeline, VM", "needs": "CUDA + cuBLAS/CUTLASS TVM build", "ready": cuda and env["USE_CUBLAS"] == "ON"},
    {"notebook": "06 / 07 runtime, profiling", "needs": "CUDA + bf16 artifact (+ checkpoint)", "ready": cuda and art and ckpt},
    {"notebook": "08 other targets", "needs": "lowering always; Metal build needs Metal codegen, Jetson export needs aarch64 g++",
     "ready": "partial" if not (env["metal codegen"] and env["aarch64 cross g++"]) else True},
])

notebook                    needs                                                                              ready
--------------------------  ---------------------------------------------------------------------------------  -----
01 eager HF                 CUDA + checkpoint                                                                  True 
02 frontend                 TVM (+ checkpoint for the real param table)                                        True 
03 parity                   CUDA + checkpoint + bf16 artifact                                                  True 
04 / 05 pipeline, VM        CUDA + cuBLAS/CUTLASS TVM build                                                    True 
06 / 07 runtime, profiling  CUDA + bf16 artifact (+ checkpoint)                                                True 
08 other targets            lowering always; Metal build needs Metal codegen, Jetson export needs aarch64 g++  True 


## Glossary

| Term | Meaning here |
|---|---|
| **block** | 32 tokens (`bd_size`) that are generated together. Attention is causal *between* blocks and bidirectional *within* one. |
| **sub-block** | 8 tokens (`small_block_size`). A block is refined one sub-block at a time. |
| **mask token** | id 151665. A block starts as masks, and each step replaces some of them with predictions. |
| **threshold** | 0.9. Each step unmasks every masked position in the current sub-block whose confidence is > 0.9, or the single best one if none is. |
| **shift** | Position *i* is predicted from the logits at *i-1* (`cat(logits[:, :1], logits[:, :-1])`). |
| **prefix cache** | Per-layer K/V of all completed blocks; extended once per finished block. |
| **block cache** | Per-layer K/V of the current block, reused by the optional partial (8-token) re-forwards. |
| **entry point** | One compiled Relax function (`embed`, `forward_cached_block`, ...). The artifact has ten. |
| **BYOC** | "Bring your own codegen": TVM hands a subgraph to a vendor library (cuBLAS, CUTLASS). |
| **dlight** | TVM's rule-based GPU scheduler for the TIR kernels that are not offloaded. |

## Takeaways
- If every row above says `True`, all notebooks can run. Notebook 08's builds are optional.
- Next: **01 · The eager HF model**.